### Informationsextraktion aus PDF (mit Llamacloud)

In [30]:
import os

from llama_cloud import LlamaCloud
from dotenv import load_dotenv

In einer .env zu hinterlegen:

LLAMA_CLOUD_API_KEY=<api_key>
GEMINI_API_KEY=<api_key>

In [31]:
load_dotenv()  # liest .env-Datei und setzt die Variablen in os.environ

True

In [3]:
client = LlamaCloud()  # reads LLAMA_CLOUD_API_KEY from the environment

In [4]:
file = client.files.create(file="Seniorenwegweiser_2025 Frankfurt (Oer).PDF", purpose="parse")

In [5]:
result = client.parsing.parse(
    file_id=file.id,
    tier="cost_effective",  # statt "agentic"
    version="latest",
    expand=["markdown"],
)

# Print the markdown for the first page
print(result.markdown.pages[0].markdown)

# Älter werden in Frankfurt (Oder)

**ODER**
**FRANKFURT**
**SŁUBICE**

Ohne Grenzen. **Bez granic.**

SENIORENWEGWEISER

www.frankfurt-oder.de


In [6]:
# Alle Seiten zu einem String zusammenfügen
markdown_content = "\n\n".join(page.markdown for page in result.markdown.pages)

# Als Markdown speichern
with open("seniorenwegweiser_fo.md", "w", encoding="utf-8") as f:
    f.write(markdown_content)

# Als Text speichern (gleicher Inhalt, .txt-Endung)
with open("seniorenwegweiser_fo.txt", "w", encoding="utf-8") as f:
    f.write(markdown_content)

## Verarbeitung des md / txt zu JSON Format (mit Google Genai)

In [15]:
import os
import json
import uuid
import time
from google import genai

In [ ]:
# TODO ergänzen um Uhrzeit und Tagesangaben --> Orientierung an JSON Schema in schema.json

# 1. fix_schema definieren (mit der Korrektur für fehlende 'items')
def fix_schema(node):
    if isinstance(node, dict):
        if "patternProperties" in node:
            inner_type = next(iter(node["patternProperties"].values()))
            return {"type": "object", "properties": {"de": fix_schema(inner_type)}}

        fixed = {}
        for key, value in node.items():
            if key in ("patternProperties", "$id"):
                continue
            if key == "properties":
                fixed[key] = {k: fix_schema(v) for k, v in value.items()}
            elif key == "items":
                fixed[key] = fix_schema(value)
            else:
                fixed[key] = value

        if fixed.get("type") == "array" and "items" not in fixed:
            fixed["items"] = {"type": "string"}

        return fixed
    return node

# 2. item_schema_fixed NEU berechnen (nutzt die aktuelle fix_schema-Version)
item_schema_fixed = fix_schema(item_schema)

# 3. DEIN BLOCK — bleibt genau so:
response_schema = {
    "type": "object",
    "properties": {
        "offers": {
            "type": "array",
            "items": item_schema_fixed,
        }
    },
    "required": ["offers"],
}

In [25]:
# --- 1. schema.json laden ---
with open("schema.json", "r", encoding="utf-8") as f:
    full_schema = json.load(f)

# Das eigentliche Item-Schema aus itemsRecord.patternProperties extrahieren
item_schema = full_schema["properties"]["itemsRecord"]["patternProperties"]["^[\\w-]+$"]

def fix_schema(node):
    """Gemini's response_schema unterstützt kein patternProperties und
    verlangt zwingend 'items' bei jedem Array-Typ. Beides wird hier
    automatisch behoben, der Rest des Schemas bleibt unverändert."""
    if isinstance(node, dict):
        if "patternProperties" in node:
            inner_type = next(iter(node["patternProperties"].values()))
            return {"type": "object", "properties": {"de": fix_schema(inner_type)}}

        fixed = {}
        for key, value in node.items():
            if key in ("patternProperties", "$id"):
                continue
            if key == "properties":
                fixed[key] = {k: fix_schema(v) for k, v in value.items()}
            elif key == "items":
                fixed[key] = fix_schema(value)
            else:
                fixed[key] = value

        # Array ohne 'items' -> Default ergänzen (z.B. bei 'tags')
        if fixed.get("type") == "array" and "items" not in fixed:
            fixed["items"] = {"type": "string"}

        return fixed
    return node

In [22]:
# --- 2. Wrapper-Schema: Liste von Angeboten ---
response_schema = {
    "type": "object",
    "properties": {
        "offers": {
            "type": "array",
            "items": item_schema_fixed,
        }
    },
    "required": ["offers"],
}

In [23]:
# --- 3. Markdown laden ---
with open("seniorenwegweiser_fo.md", "r", encoding="utf-8") as f:
    md_content = f.read()

In [28]:
# --- 4. Extraktion via Gemini (ein Call für alle Angebote) ---
client = genai.Client(api_key=os.environ.get("GEMINI_API_KEY"))

response = client.models.generate_content(
    model="gemini-3.5-flash-lite",
    contents=f"""Das folgende Dokument enthält MEHRERE Angebote für
Senior:innen in Brandenburg (z.B. verschiedene Einrichtungen, Kurse
oder Beratungsstellen).

Identifiziere JEDES einzelne Angebot im Dokument und extrahiere für
JEDES Angebot separat die passenden Informationen gemäß dem
vorgegebenen Schema. Fülle nur Felder aus, für die im Text
tatsächlich Informationen vorhanden sind, lasse den Rest weg.
Nutze bei mehrsprachigen Feldern (brief, description, etc.) nur "de".
Setze 'state' immer auf "draft".
Schreibe verständlich, ohne Fachjargon.

DOKUMENT:
{md_content}""",
    config={
        "response_mime_type": "application/json",
        "response_schema": response_schema,
    },
)

result = json.loads(response.text)
offers = result["offers"]
print(f"{len(offers)} Angebote gefunden")

136 Angebote gefunden


In [29]:
# --- 5. In vollständiges AdapterData-Format einbetten ---
items_record = {str(uuid.uuid4()): offer for offer in offers}

adapter_data = {
    "adapter": {
        "name": "senior-offers-pdf-import",
        "sourceName": "PDF Import",
    },
    "lastUpdate": int(time.time()),
    "version": "1.0",
    "itemsRecord": items_record,
}

# --- 6. Speichern ---
with open("output.json", "w", encoding="utf-8") as f:
    json.dump(adapter_data, f, ensure_ascii=False, indent=2)

print(f"{len(items_record)} Items in output.json gespeichert")

136 Items in output.json gespeichert
